# Imports and setup

In [1]:
import sys
sys.path.append('../..')


In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import h5py
import torch
from pathlib import Path
import kaleido
kaleido.get_chrome_sync()

from realtime_sim.ctc_decoder import greedy_decode_batch

In [3]:
sns.set_theme(font='Arial', font_scale=1.2, style='white', palette=px.colors.qualitative.Plotly)
# sns.set_theme(font='Arial', font_scale=1.2, style='white', palette=sns.color_palette("cubehelix"))
sns.set_style(rc={
    # 'axes.edgecolor': 'gray',
    'axes.linewidth': 0.5,
    'xtick.bottom': True,
    'ytick.left': True,
    'svg.fonttype': 'none'
    })
plt.rcParams['svg.fonttype'] = 'none'

In [4]:
pt_names = ['S14', 'S22', 'S23', 'S26', 'S33', 'S39', 'S58', 'S62']
DATA_DIR = Path('../../data/')
paper_pt_dict = {'S14': 'S1', 'S26': 'S2', 'S33': 'S3', 'S22': 'S4', 'S23': 'S5', 'S39': 'S6', 'S58': 'S7', 'S62': 'S8'}
FIG_DIR = Path('~/Box/Coganlab/Papers/2025/cross-patient-speech-decoding/figures/supp/5b_all_pts').expanduser()
ctc_res_dir = DATA_DIR / 'results' / 'decoding' / 'ctc_results_90varNoDel_computeAlign_augs'

In [5]:
PHON_DICT = {
    0: 'blank',
    1: 'a',
    2: 'ae',
    3: 'i',
    4: 'u',
    5: 'b',
    6: 'p',
    7: 'v',
    8: 'g',
    9: 'k',
}

# CTC decoding all patients

In [6]:
res_pts = ['S14', 'S26', 'S22', 'S23', 'S39', 'S58', 'S62']
contexts = {'Chance': '_chance', 'Patient-specific': '_ptSpecific', 'Unaligned': '_unaligned', 'Aligned': '_aligned'}
tw = [0.5, 3.5]

ctc_decode_df = pd.DataFrame()
for pt in res_pts:
    skip_outer = False

    pers = np.empty((len(contexts), 100)) * np.nan
    for i, (context, c_suffix) in enumerate(contexts.items()):
        res_path = ctc_res_dir / f'{pt}/{pt}_ctcRNN_decodeTW([{tw[0]},{tw[1]}]){c_suffix}.h5'
        try:
            with h5py.File(res_path, 'r') as f:
                pers_data = f['phoneme_error_rate'][:]
                pers[i, :len(pers_data)] = pers_data
        except FileNotFoundError:
            print(f'{str(res_path)} not found. Skipping patient {pt}.')
            skip_outer=True
            # break
    
    if skip_outer:
        continue
        
    pt_df = pd.DataFrame(pers.T, columns=list(contexts.keys()))
    pt_df = pt_df.dropna()
    pt_df = pt_df.melt(var_name='Decoding type', value_name='Phoneme error rate')
    pt_df['Patient'] = paper_pt_dict[pt]
    ctc_decode_df = pd.concat([ctc_decode_df, pt_df], axis=0)
ctc_decode_df.reset_index(drop=True, inplace=True)
ctc_decode_df

,Decoding type,Phoneme error rate,Patient
0,Chance,86.538462,S1
1,Chance,83.333333,S1
2,Chance,84.615385,S1
3,Chance,84.615385,S1
4,Chance,87.179487,S1
...,...,...,...
1395,Aligned,86.538462,S8
1396,Aligned,83.333333,S8
1397,Aligned,79.487179,S8
1398,Aligned,82.051282,S8


In [7]:
SCALE_FACTOR = 0.28
fig_df = ctc_decode_df.copy()

fig = px.box(fig_df, x='Patient', y="Phoneme error rate", color='Decoding type',
             points='all', width=1800, height=600,
             color_discrete_sequence=['#7f8080', '#636EFA', '#EF553B', '#AB63FA'])
fig.update_traces(marker=dict(
                            #   size=2,
                              opacity=0.5,
                              ),
                  line=dict(width=1/SCALE_FACTOR))

fig.update_layout(
    plot_bgcolor='white',
    legend=dict(
        orientation='h',
        yanchor='top',
        y=1.03,
        xanchor='left',
        x=0.02,
        title_text=''
    ),
    title_text='',
    title_x=0.5,
    font=dict(size=18, family='Arial', color='black'),
    boxgroupgap=0.4,
)
fig.update_xaxes(
    title='',
    mirror=False,
    ticks='outside',
    showline=True,
    linecolor='black',
    tickcolor='black',
    linewidth=1/SCALE_FACTOR,
    tickwidth=1/SCALE_FACTOR,
    showgrid=False,
)
fig.update_yaxes(
    title='Phoneme error rate (%)',
    mirror=False,
    ticks='outside',
    showline=True,
    linecolor='black',
    tickcolor='black',
    linewidth=1/SCALE_FACTOR,
    tickwidth=1/SCALE_FACTOR,
    showgrid=False,
    range=[50, 110]
    # range=[20, 70]
)

fig.write_image(FIG_DIR / 'fig_5b_all_pts.svg', scale=SCALE_FACTOR)
fig.show()

In [10]:
pts = [p for p in paper_pt_dict.values() if p != 'S3']
p_arr = np.zeros((len(pts), 3))
for i, pt in enumerate(pts):
    pt_chance = ctc_decode_df[(ctc_decode_df['Patient'] == pt) & (ctc_decode_df['Decoding type'] == 'Chance')]['Phoneme error rate']
    pt_spf = ctc_decode_df[(ctc_decode_df['Patient'] == pt) & (ctc_decode_df['Decoding type'] == 'Patient-specific')]['Phoneme error rate']
    pt_una = ctc_decode_df[(ctc_decode_df['Patient'] == pt) & (ctc_decode_df['Decoding type'] == 'Unaligned')]['Phoneme error rate']
    pt_cca = ctc_decode_df[(ctc_decode_df['Patient'] == pt) & (ctc_decode_df['Decoding type'] == 'Aligned')]['Phoneme error rate']

    stat, anova_p = stats.f_oneway(pt_chance, pt_spf, pt_una, pt_cca)
    res = stats.tukey_hsd(pt_chance, pt_spf, pt_una, pt_cca)
    chance_ps_p = res.pvalue[0,1]
    chance_una_p = res.pvalue[0,2]
    chance_cca_p = res.pvalue[0,3]
    ps_una_p = res.pvalue[1,2]
    ps_cca_p = res.pvalue[1,3]
    una_cca_p = res.pvalue[2,3]

    print(f'{pt}: ANOVA p = {anova_p}, Chance vs. PS p = {chance_ps_p}, Chance vs. Unaligned p = {chance_una_p}, Chance vs. Aligned p = {chance_cca_p}, PS vs. Unaligned p = {ps_una_p}, PS vs. Aligned p = {ps_cca_p}, Unaligned vs. Aligned p = {una_cca_p}')
    print(f'ANOVA F-statistic: {stat}')
    print(res)

S1: ANOVA p = 2.766996670982547e-55, Chance vs. PS p = 4.6629367034256575e-14, Chance vs. Unaligned p = 4.6629367034256575e-14, Chance vs. Aligned p = 4.6629367034256575e-14, PS vs. Unaligned p = 2.009570287953011e-11, PS vs. Aligned p = 4.696243394164412e-14, Unaligned vs. Aligned p = 0.3055704151024313
ANOVA F-statistic: 175.58389438018463
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      8.846     0.000     6.842    10.850
 (0 - 2)     14.590     0.000    12.586    16.594
 (0 - 3)     15.936     0.000    13.932    17.940
 (1 - 0)     -8.846     0.000   -10.850    -6.842
 (1 - 2)      5.744     0.000     3.740     7.748
 (1 - 3)      7.090     0.000     5.086     9.094
 (2 - 0)    -14.590     0.000   -16.594   -12.586
 (2 - 1)     -5.744     0.000    -7.748    -3.740
 (2 - 3)      1.346     0.306    -0.658     3.350
 (3 - 0)    -15.936     0.000   -17.940   -13.932
 (3 - 1)     -7.090     0.000    -9.09